In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import mysql.connector
from sqlalchemy import create_engine


ModuleNotFoundError: No module named 'pandas'

In [ ]:
pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Create SQLAlchemy engine with the correct credentials
engine = create_engine('mysql+mysqlconnector://root:Yasmine123/@localhost:3308/final_shema')

NameError: name 'create_engine' is not defined

In [145]:
# list of columns we care about
COLUMNS = [
    'RecipeId', 'NameR', 'CookTime', 'PrepTime', 'TotalTime', 'RecipeIngredientParts',
    'Calories', 'FatContent', 'SaturatedFatContent', 'CholesterolContent',
    'SodiumContent', 'CarbohydrateContent', 'FiberContent', 'SugarContent',
    'ProteinContent', 'RecipeInstructions'
]


In [147]:
# 2) load & preprocess once
def load_and_preprocess_recipes():
    # load entire table
    df = pd.read_sql('SELECT * FROM meal', engine)[COLUMNS]
    # clean
    df.dropna(inplace=True)
    df.drop_duplicates(inplace=True)
    # nutritional filters
    max_list = [2000,100,13,300,2300,300,25,50,50]
    nutrient_cols = df.columns[6:15]
    for nut,limit in zip(nutrient_cols, max_list):
        df = df[df[nut] < limit]
    return df


In [149]:

# 3) content-based
def get_recommendations(user_preferences: str, extracted_data: pd.DataFrame, top_n=5):
    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(extracted_data['RecipeIngredientParts'].astype(str))
    user_vec = tfidf.transform([user_preferences])
    sim = cosine_similarity(user_vec, tfidf_matrix)[0]
    idx = sim.argsort()[-top_n:][::-1]
    return extracted_data.iloc[idx]

In [151]:
# 4) collaborative
def meal_distribution(n):
    if n==3: return [0.3,0.4,0.3]
    if n==5: return [0.25,0.1,0.35,0.1,0.2]
    return [1/n]*n


In [153]:
def collaborative_filtering(user_age, user_height, user_weight, meals_per_day, gender, extracted_data: pd.DataFrame, top_n=5):
    # ensure numeric
    user_age, user_height, user_weight = map(float,[user_age, user_height, user_weight])
    # BMR
    if gender=='male':
        daily = 10*user_weight + 6.25*user_height - 5*user_age + 5
    else:
        daily = 10*user_weight + 6.25*user_height - 5*user_age - 161
    dist = meal_distribution(meals_per_day)
    recs = {}
    for i,p in enumerate(dist):
        cap = daily * p
        filt = extracted_data[extracted_data['Calories'] <= cap] \
                   .sort_values('ProteinContent',ascending=False)
        recs[f"Meal {i+1}"] = filt.head(top_n)
    return recs

In [155]:
# DB helpers
def get_db_connection():
    db = mysql.connector.connect(host="127.0.0.1",user="root",password="ouissem",database="final_shema")
    return db, db.cursor()

In [157]:
def get_user_info(user_id):
    db,cur = get_db_connection()
    cur.execute("SELECT age, taille, poids, objectif, sexe, meals_perday, preferences FROM user WHERE idUser=%s",(user_id,))
    row = cur.fetchone()
    db.close()
    return row



In [177]:
def insert_recommended_meals(user_id, ids):
    db,cur = get_db_connection()
    for rid in ids:
        cur.execute(
          "INSERT INTO recommende_meals (idUser, RecipeId, recommended_at) VALUES (%s,%s,NOW())",
          (user_id,rid)
        )
    db.commit()
    db.close()

In [179]:
# 5) hybrid
def hybrid_recommendation(user_id, top_n=5):
    # ← use the parameter user_id here, not idUser
    ui = get_user_info(user_id)
    if ui is None:
        raise ValueError(f"user {user_id} not found")
    age, height, weight, goal, gender, meals_per_day, prefs = ui

    data = load_and_preprocess_recipes()
    content = get_recommendations(prefs, data, top_n)
    collab = collaborative_filtering(age, height, weight, meals_per_day, gender, data, top_n)

    hybrid = {}
    all_ids = []
    for key, df2 in collab.items():
        df1 = content.sample(n=top_n, replace=True)
        dfh = pd.concat([df1, df2]).drop_duplicates().sample(frac=1).head(top_n)
        hybrid[key] = dfh
        all_ids += dfh['RecipeId'].tolist()

    insert_recommended_meals(user_id, all_ids)
    return hybrid


In [185]:
# —— test in same file ——
if __name__=="__main__":
    out = hybrid_recommendation(1, top_n=5)
    for meal,df in out.items():
        print(f"\n{meal}")
        print(df[['RecipeId','NameR','Calories','ProteinContent']])


Meal 1
        RecipeId                                    NameR  Calories  \
194074    241217  Weight Watcher's Turkey Sausage Patties      68.4   
202962    252638         Breakfast Turkey Sausage Patties      68.0   
150669    188149    Steamed Cod With Ginger and Scallions     155.6   
422398    520960  6WBM Breakfast Sausage-Jimmy Dean Style      67.2   
150670    188150                     Best Tuna Melts Ever     293.1   

        ProteinContent  
194074            14.4  
202962            14.4  
150669            31.9  
422398            14.1  
150670            25.7  

Meal 2
        RecipeId                                           NameR  Calories  \
439968    541379                  Meg's Fresh Ginger Gingerbread     316.6   
194074    241217         Weight Watcher's Turkey Sausage Patties      68.4   
150668    188148                          Maple Upside Down Cake     305.1   
422398    520960         6WBM Breakfast Sausage-Jimmy Dean Style      67.2   
422733    521329 